# Hybrid-Approach Seasonal Projection Differences — SSP5-8.5
## Notebook 21

Identical logic to Notebook 16 (SSP2-4.5). Reads NB19 (3-model) and raw single-model NC files for SSP5-8.5, applies the same per-basin best-approach assignment, and saves seasonal mean + difference NC files.

**Outputs → `ssp5-8.5 output\projections\`**
```
seasonal_means/
    hybrid_ssp585_ref_wet.nc        (mm/month, 1995-2014, Oct-Mar)
    hybrid_ssp585_ref_dry.nc        (mm/month, 1995-2014, Apr-Sep)
    hybrid_ssp585_near_wet.nc       (mm/month, 2021-2040)
    hybrid_ssp585_near_dry.nc
    hybrid_ssp585_mid_wet.nc        (mm/month, 2041-2060)
    hybrid_ssp585_mid_dry.nc
differences/
    hybrid_ssp585_near_wet_diff.nc  (near - ref, mm/month)
    hybrid_ssp585_near_dry_diff.nc
    hybrid_ssp585_mid_wet_diff.nc
    hybrid_ssp585_mid_dry_diff.nc
```

## 1. Imports

In [1]:
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")
print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")

Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Paths & Settings

In [2]:
COMMENTS_ROOT = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments")
OUTPUT_ROOT   = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output")

SHAPEFILE          = COMMENTS_ROOT / r"jordan.basins\surface.basins.for.jordan\Jo.shp"
RANKINGS_CSV       = COMMENTS_ROOT / r"validation\single.model\basin_model_rankings.csv"
RECOMMENDATION_CSV = COMMENTS_ROOT / r"validation\comparison\basin_approach_recommendation.csv"

# NB19 Jordan-wide 3-model ensemble NC files for SSP5-8.5
ENS3_DIR = OUTPUT_ROOT / "3 ensemble models"
# Pattern: Jordan_3model_ensemble_ssp585_{nc_label}.nc

# Raw single-model NC files for SSP5-8.5 (filename prefix: merged_)
MODEL_DIR = Path(r"D:\RICAAR8.5\merge\Precipitation")
# Pattern: merged_{ModelName}.nc   Variable: prAdjust (mm/day)

SEASONAL_DIR = OUTPUT_ROOT / "projections" / "seasonal_means"
DIFF_DIR     = OUTPUT_ROOT / "projections" / "differences"
SEASONAL_DIR.mkdir(parents=True, exist_ok=True)
DIFF_DIR.mkdir(parents=True, exist_ok=True)

PR_VAR       = "prAdjust"
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

# Period labels — must match NB19 output filenames exactly
PERIODS = {
    "ref" : {"nc_label": "1995_2014", "years": (1995, 2014)},
    "near": {"nc_label": "2021_2040", "years": (2021, 2040)},
    "mid" : {"nc_label": "2041_2060", "years": (2041, 2060)},
}

WET_MONTHS = [10, 11, 12, 1, 2, 3]
DRY_MONTHS = [4, 5, 6, 7, 8, 9]
SEASONS    = {"wet": WET_MONTHS, "dry": DRY_MONTHS}

print("Paths configured.")
print(f"  ENS3 dir   : {ENS3_DIR}")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Output     : {OUTPUT_ROOT / 'projections'}")

Paths configured.
  ENS3 dir   : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output\3 ensemble models
  Model dir  : D:\RICAAR8.5\merge\Precipitation
  Output     : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ssp5-8.5 output\projections


## 3. Load Rankings and Approach Recommendations

In [3]:
rankings = pd.read_csv(RANKINGS_CSV)
rankings.columns = [c.strip() for c in rankings.columns]

top3_per_basin = (
    rankings.sort_values(["Basin", "Avg_Rank"])
    .groupby("Basin")["Model"]
    .apply(lambda x: x.head(3).tolist())
    .to_dict()
)
best_single_per_basin = (
    rankings.sort_values(["Basin", "Avg_Rank"])
    .groupby("Basin")["Model"]
    .first()
    .to_dict()
)

rec_df = pd.read_csv(RECOMMENDATION_CSV)
rec_df.columns = [c.strip() for c in rec_df.columns]
LABEL_TO_CODE  = {"Best Single Model": "single", "3-Model Ensemble": "ens3"}
basin_approach = {
    row["Basin"]: LABEL_TO_CODE[row["Recommended_Approach"]]
    for _, row in rec_df.iterrows()
    if row["Recommended_Approach"] in LABEL_TO_CODE
}

print("Approach per validated basin (NB11):")
for b, a in basin_approach.items():
    extra = f"  [{best_single_per_basin.get(b)}]" if a == "single" else ""
    print(f"  {b:<30} → {a}{extra}")

Approach per validated basin (NB11):
  N.R.S.W                        → ens3
  YARMOUK (JORDAN)               → single  [MPI-ESM1-2-LR]
  JORDAN VALLY (JORDAN)          → single  [MPI-ESM1-2-LR]
  AMMAN ZARQA (JORDAN)           → single  [MPI-ESM1-2-LR]
  S.R.S.W                        → single  [MPI-ESM1-2-LR]
  D.S.R.S.W                      → ens3
  MUJIB                          → single  [MPI-ESM1-2-LR]
  W. ARABA NORTH                 → ens3
  HASA                           → single  [MPI-ESM1-2-LR]
  AZRAQ (JORDAN)                 → ens3
  JAFER                          → single  [NorESM2-MM]
  HAMMAD                         → single  [MPI-ESM1-2-LR]


## 4. Load Shapefile and KD-Tree Assignment for Unranked Basins

In [4]:
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy().reset_index(drop=True)
gdf_jordan["cx"] = gdf_jordan.geometry.centroid.x
gdf_jordan["cy"] = gdf_jordan.geometry.centroid.y

ranked_rows   = [r for _, r in gdf_jordan.iterrows() if r["BASIN_NAME"] in basin_approach]
ranked_coords = np.array([[r["cy"], r["cx"]] for r in ranked_rows])
ranked_names  = [r["BASIN_NAME"] for r in ranked_rows]
kd_tree       = cKDTree(ranked_coords)

for _, row in gdf_jordan.iterrows():
    b = row["BASIN_NAME"]
    if b not in basin_approach:
        dist_deg, idx = kd_tree.query([row["cy"], row["cx"]])
        nearest = ranked_names[idx]
        basin_approach[b]        = basin_approach[nearest]
        top3_per_basin[b]        = top3_per_basin[nearest]
        best_single_per_basin[b] = best_single_per_basin[nearest]
        print(f"  KD-tree: {b:<35} → '{basin_approach[b]}'  (nearest: {nearest}, {dist_deg*111.32:.1f} km)")

print()
print(f"Total Jordan basins assigned: {len(basin_approach)}")

  KD-tree: J.VALLEY-YARMOUK TRIANGLE           → 'ens3'  (nearest: N.R.S.W, 29.2 km)
  KD-tree: WADI ARABA SOUTH                    → 'ens3'  (nearest: W. ARABA NORTH, 100.5 km)
  KD-tree: QA DISI & SOUTHERN DESERT           → 'single'  (nearest: JAFER, 74.3 km)
  KD-tree: SARHAN                              → 'ens3'  (nearest: AZRAQ (JORDAN), 119.8 km)

Total Jordan basins assigned: 16


## 5. Load basin_id Grid from NB19 Output

In [5]:
sample_nc  = ENS3_DIR / "Jordan_3model_ensemble_ssp585_1995_2014.nc"
ds_sample  = xr.open_dataset(sample_nc)

lats          = ds_sample["lat"].values
lons          = ds_sample["lon"].values
n_lat, n_lon  = len(lats), len(lons)
basin_id_grid = ds_sample["basin_id"].values.astype(np.int8)

# Parse basin_id_labels: format "0=HAMMAD; 1=YARMOUK (JORDAN); ..."
labels_str  = ds_sample["basin_id"].attrs["basin_id_labels"]
idx_to_name = {}
for part in labels_str.split(";"):
    part = part.strip()
    if "=" in part:
        idx_s, bname = part.split("=", 1)
        idx_to_name[int(idx_s.strip())] = bname.strip()

ds_sample.close()

print(f"Grid       : {n_lat} lat × {n_lon} lon")
print(f"Inside JO  : {int((basin_id_grid >= 0).sum())} cells")
print(f"Outside JO : {int((basin_id_grid < 0).sum())} cells")
print()
print(f"{'idx':<4} {'Basin':<35} {'Approach'}")
print("-" * 55)
for idx in sorted(idx_to_name):
    b = idx_to_name[idx]
    a = basin_approach.get(b, "?")
    n = int((basin_id_grid == idx).sum())
    print(f"  {idx:>2}  {b:<33} {a:<8}  [{n} cells]")

Grid       : 85 lat × 75 lon
Inside JO  : 6375 cells
Outside JO : 0 cells

idx  Basin                               Approach
-------------------------------------------------------
   0  HAMMAD                            single    [5711 cells]
   1  YARMOUK (JORDAN)                  single    [11 cells]
   2  J.VALLEY-YARMOUK TRIANGLE         ens3      [0 cells]
   3  JORDAN VALLY (JORDAN)             single    [5 cells]
   4  N.R.S.W                           ens3      [11 cells]
   5  AZRAQ (JORDAN)                    ens3      [113 cells]
   6  AMMAN ZARQA (JORDAN)              single    [33 cells]
   7  S.R.S.W                           single    [6 cells]
   8  MUJIB                             single    [61 cells]
   9  D.S.R.S.W                         ens3      [15 cells]
  10  W. ARABA NORTH                    ens3      [27 cells]
  11  HASA                              single    [22 cells]
  12  JAFER                             single    [116 cells]
  13  WADI ARABA SOUTH   

## 6. Helper Functions

In [6]:
def seasonal_mean_from_nc(nc_path, season_months):
    """
    Load a Jordan-wide NC file (prAdjust, mm/day), compute standardised monthly
    totals (30/D_m × sum_daily), return long-term seasonal mean (mm/month).
    """
    ds      = xr.open_dataset(nc_path)
    da      = ds[PR_VAR]
    da_msum = da.resample(time="1MS").sum(skipna=True)
    da_mstd = da_msum * (30.0 / da_msum["time"].dt.days_in_month)
    mask    = da_mstd["time"].dt.month.isin(season_months)
    result  = da_mstd.sel(time=mask).mean(dim="time", skipna=True).values
    ds.close()
    return result.astype(np.float32)


def single_model_seasonal_mean(model_name, year_range, season_months):
    """
    Same but for a raw SSP5-8.5 single-model file (merged_{model}.nc),
    sliced to year_range.
    """
    nc_path = MODEL_DIR / f"merged_{model_name}.nc"
    ds      = xr.open_dataset(nc_path)
    da      = ds[PR_VAR].sel(time=slice(str(year_range[0]), str(year_range[1])))
    da_msum = da.resample(time="1MS").sum(skipna=True)
    da_mstd = da_msum * (30.0 / da_msum["time"].dt.days_in_month)
    mask    = da_mstd["time"].dt.month.isin(season_months)
    result  = da_mstd.sel(time=mask).mean(dim="time", skipna=True).values
    ds.close()
    return result.astype(np.float32)


print("Helper functions defined.")

Helper functions defined.


## 7. Build and Save Hybrid Seasonal Mean NC Files

In [7]:
def build_and_save_seasonal_mean(period_key, season_key):
    pcfg   = PERIODS[period_key]
    p_lbl  = pcfg["nc_label"]
    p_yrs  = pcfg["years"]
    s_mths = SEASONS[season_key]

    outfile = SEASONAL_DIR / f"hybrid_ssp585_{period_key}_{season_key}.nc"
    if outfile.exists():
        print(f"  [SKIP — exists] {outfile.name}")
        return outfile

    print(f"\nBuilding: {period_key} ({p_lbl})  |  season: {season_key}")

    # Load 3-model seasonal mean from NB19 output
    nc3 = ENS3_DIR / f"Jordan_3model_ensemble_ssp585_{p_lbl}.nc"
    print(f"  ens3  ← {nc3.name}")
    data_ens3 = seasonal_mean_from_nc(nc3, s_mths)

    # Load single-model seasonal means — only models needed by single basins
    needed_models = {
        best_single_per_basin[b]
        for b in basin_approach
        if basin_approach[b] == "single" and b in best_single_per_basin
    }
    data_single = {}
    for model in sorted(needed_models):
        print(f"  single ← {model}")
        data_single[model] = single_model_seasonal_mean(model, p_yrs, s_mths)

    # Assemble hybrid grid
    print("  Assembling hybrid grid ...")
    hybrid = np.full((n_lat, n_lon), np.nan, dtype=np.float32)

    for i in range(n_lat):
        for j in range(n_lon):
            cell_id = int(basin_id_grid[i, j])
            if cell_id < 0:
                continue
            bname    = idx_to_name.get(cell_id)
            approach = basin_approach.get(bname, "ens3")

            if approach == "single":
                model = best_single_per_basin.get(bname)
                hybrid[i, j] = data_single.get(model, data_ens3)[i, j]
            else:
                hybrid[i, j] = data_ens3[i, j]

    ds_out = xr.Dataset(
        {"pr": xr.DataArray(
            hybrid, dims=["lat", "lon"],
            attrs={
                "long_name"    : "Hybrid-approach seasonal mean (30-day standardised)",
                "units"        : "mm/month",
                "season"       : season_key,
                "season_months": str(s_mths),
                "period"       : f"{p_yrs[0]}-{p_yrs[1]}",
                "scenario"     : "SSP5-8.5",
                "approach"     : "hybrid — per-basin best approach from NB11",
            }
        )},
        coords={
            "lat": ("lat", lats, {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}),
            "lon": ("lon", lons, {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}),
        },
        attrs={
            "title"      : f"Hybrid seasonal mean SSP5-8.5 {period_key} {season_key}",
            "Conventions": "CF-1.7",
            "created"    : str(datetime.datetime.now()),
            "source"     : "RICAAR SMHI-HCLIM-ALADIN-38, SMHI-MIdAS021",
        }
    )
    encoding = {"pr": {"zlib": True, "complevel": 4, "dtype": "float32",
                       "_FillValue": np.float32(1e+20)}}
    ds_out.to_netcdf(outfile, encoding=encoding, format="NETCDF4")

    v = hybrid[~np.isnan(hybrid)]
    print(f"  Saved → {outfile.name}")
    print(f"    cells={v.size}  mean={v.mean():.2f}  max={v.max():.2f}  min={v.min():.2f} mm/month")
    return outfile


print("build_and_save_seasonal_mean() defined.")

build_and_save_seasonal_mean() defined.


## 8. Run — All 6 Seasonal Mean Files

In [8]:
seasonal_files = {}

for period_key in ["ref", "near", "mid"]:
    for season_key in ["wet", "dry"]:
        path = build_and_save_seasonal_mean(period_key, season_key)
        seasonal_files[(period_key, season_key)] = path

print("\n" + "=" * 55)
print("All 6 seasonal mean files complete.")
print("=" * 55)


Building: ref (1995_2014)  |  season: wet
  ens3  ← Jordan_3model_ensemble_ssp585_1995_2014.nc
  single ← MPI-ESM1-2-LR
  single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp585_ref_wet.nc
    cells=6375  mean=13.64  max=139.69  min=0.00 mm/month

Building: ref (1995_2014)  |  season: dry
  ens3  ← Jordan_3model_ensemble_ssp585_1995_2014.nc
  single ← MPI-ESM1-2-LR
  single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp585_ref_dry.nc
    cells=6375  mean=2.19  max=23.37  min=0.00 mm/month

Building: near (2021_2040)  |  season: wet
  ens3  ← Jordan_3model_ensemble_ssp585_2021_2040.nc
  single ← MPI-ESM1-2-LR
  single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp585_near_wet.nc
    cells=6375  mean=13.42  max=142.47  min=0.00 mm/month

Building: near (2021_2040)  |  season: dry
  ens3  ← Jordan_3model_ensemble_ssp585_2021_2040.nc
  single ← MPI-ESM1-2-LR
  single ← NorESM2-MM
  Assembling hybrid grid ...
  Saved → hybrid_ssp585_near_dry

## 9. Compute Differences (Future − Reference) and Save

In [9]:
PERIOD_TITLES = {"near": "2021-2040", "mid": "2041-2060"}

ref_grids = {}
for season_key in ["wet", "dry"]:
    ds = xr.open_dataset(seasonal_files[("ref", season_key)])
    ref_grids[season_key] = ds["pr"].values.astype(np.float32)
    ds.close()

diff_files = {}

for future_period in ["near", "mid"]:
    for season_key in ["wet", "dry"]:
        outfile = DIFF_DIR / f"hybrid_ssp585_{future_period}_{season_key}_diff.nc"

        if outfile.exists():
            print(f"[SKIP — exists] {outfile.name}")
            diff_files[(future_period, season_key)] = outfile
            continue

        ds_fut = xr.open_dataset(seasonal_files[(future_period, season_key)])
        pr_fut = ds_fut["pr"].values.astype(np.float32)
        pr_ref = ref_grids[season_key]
        delta  = pr_fut - pr_ref
        ds_fut.close()

        ds_diff = xr.Dataset(
            {
                "pr_diff": xr.DataArray(
                    delta, dims=["lat", "lon"],
                    attrs={
                        "long_name"    : f"Change in monthly precip: {PERIOD_TITLES[future_period]} minus 1995-2014",
                        "units"        : "mm/month",
                        "scenario"     : "SSP5-8.5",
                        "approach"     : "hybrid",
                        "future_period": PERIOD_TITLES[future_period],
                        "reference"    : "1995-2014",
                        "season"       : season_key,
                    }
                ),
                "pr_ref": xr.DataArray(
                    pr_ref, dims=["lat", "lon"],
                    attrs={"long_name": "Reference (1995-2014) seasonal mean",
                           "units": "mm/month"}
                ),
                "pr_future": xr.DataArray(
                    pr_fut, dims=["lat", "lon"],
                    attrs={"long_name": f"{PERIOD_TITLES[future_period]} seasonal mean",
                           "units": "mm/month"}
                ),
            },
            coords={
                "lat": ("lat", lats, {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}),
                "lon": ("lon", lons, {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}),
            },
            attrs={
                "title"      : f"Hybrid precip change SSP5-8.5 {future_period} {season_key}",
                "Conventions": "CF-1.7",
                "created"    : str(datetime.datetime.now()),
            }
        )

        enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32",
                   "_FillValue": np.float32(1e+20)}
               for v in ["pr_diff", "pr_ref", "pr_future"]}
        ds_diff.to_netcdf(outfile, encoding=enc, format="NETCDF4")
        diff_files[(future_period, season_key)] = outfile

        v = delta[~np.isnan(delta)]
        print(f"{outfile.name}")
        print(f"  mean={v.mean():.2f}  max={v.max():.2f}  min={v.min():.2f}  points={v.size}")

print("\nAll 4 difference NC files saved.")

hybrid_ssp585_near_wet_diff.nc
  mean=-0.21  max=7.74  min=-13.27  points=6375
hybrid_ssp585_near_dry_diff.nc
  mean=0.08  max=3.89  min=-2.25  points=6375
hybrid_ssp585_mid_wet_diff.nc
  mean=0.23  max=9.50  min=-12.59  points=6375
hybrid_ssp585_mid_dry_diff.nc
  mean=0.33  max=2.22  min=-5.49  points=6375

All 4 difference NC files saved.


## 10. Verify All Outputs

In [10]:
print("=" * 72)
print("SEASONAL MEAN FILES  →  projections/seasonal_means/")
print("=" * 72)
for f in sorted(SEASONAL_DIR.glob("hybrid_ssp585_*.nc")):
    ds = xr.open_dataset(f)
    pr = ds["pr"].values
    v  = pr[~np.isnan(pr)]
    sz = f.stat().st_size / 1024
    print(f"  {f.name:<46} cells={v.size:>3} "
          f" mean={v.mean():>6.2f}  max={v.max():>7.2f}  min={v.min():>5.2f} mm/mo  [{sz:.0f} KB]")
    ds.close()

print()
print("=" * 72)
print("DIFFERENCE FILES     →  projections/differences/")
print("=" * 72)
for f in sorted(DIFF_DIR.glob("hybrid_ssp585_*.nc")):
    ds = xr.open_dataset(f)
    d  = ds["pr_diff"].values
    v  = d[~np.isnan(d)]
    sz = f.stat().st_size / 1024
    print(f"  {f.name:<52} mean={v.mean():>6.2f}  max={v.max():>6.2f}  min={v.min():>6.2f}  [{sz:.0f} KB]")
    ds.close()

print()
print("Ready for Notebook 22 — Projection Maps.")

SEASONAL MEAN FILES  →  projections/seasonal_means/
  hybrid_ssp585_mid_dry.nc                       cells=6375  mean=  2.52  max=  18.50  min= 0.00 mm/mo  [34 KB]
  hybrid_ssp585_mid_wet.nc                       cells=6375  mean= 13.86  max= 143.96  min= 0.00 mm/mo  [34 KB]
  hybrid_ssp585_near_dry.nc                      cells=6375  mean=  2.27  max=  23.89  min= 0.00 mm/mo  [34 KB]
  hybrid_ssp585_near_wet.nc                      cells=6375  mean= 13.42  max= 142.47  min= 0.00 mm/mo  [34 KB]
  hybrid_ssp585_ref_dry.nc                       cells=6375  mean=  2.19  max=  23.37  min= 0.00 mm/mo  [34 KB]
  hybrid_ssp585_ref_wet.nc                       cells=6375  mean= 13.64  max= 139.69  min= 0.00 mm/mo  [34 KB]

DIFFERENCE FILES     →  projections/differences/
  hybrid_ssp585_mid_dry_diff.nc                        mean=  0.33  max=  2.22  min= -5.49  [77 KB]
  hybrid_ssp585_mid_wet_diff.nc                        mean=  0.23  max=  9.50  min=-12.59  [77 KB]
  hybrid_ssp585_near_dry_d